<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/01_hero.svg" align="center" width="80%">
</div>

<br>

# BYTE-PAIR ENCODING BY HAND

<br>

**About:** Build a byte-pair-encoding tokenizer from a 30-word corpus in plain Python, then use it to see why a token, not a word, is the unit a language model reads and bills.

**Learning Goals:**
- Explain why a language model reads token ids rather than characters or words, and what that costs in context, price and latency
- Build a byte-pair-encoding vocabulary by hand: count adjacent pairs, merge the most frequent, repeat for N merges
- Encode a new sentence with the learned merges, decode the ids back, and locate where an unseen word splits into pieces
- Measure how vocabulary size trades off against sequence length on the same text

**Keywords:** tokenization, byte-pair encoding, bpe, token ids, subword units, merges, vocabulary size, sequence length

**Prerequisite Knowledge:** (1) Python basics: lists, dictionaries and loops, (2) What a language model is, at the level of "it predicts the next piece of text"

**Target User:** A practitioner or decision maker who has used an LLM and wants to understand why prompts are billed in tokens and why some words are "expensive".

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: WHY MODELS READ NUMBERS](#Part_1)
> #### [PART 2: COUNTING PAIRS AND MERGING](#Part_2)
> #### [PART 3: ENCODING AND DECODING](#Part_3)
> #### [PART 4: VOCABULARY SIZE AGAINST SEQUENCE LENGTH](#Part_4)

<br>

In [ ]:
# Setup - imports and environment check
import os
import re
import sys
from collections import Counter

import matplotlib.pyplot as plt

os.makedirs('images', exist_ok=True)  # Part 4 saves its plot here

print(f'Python: {sys.version.split()[0]}')


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## Why models read **numbers**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/01_01_diagram.svg" align="center" width="50%" padding="10"><br>
    <br>
</div>

PART 1 - Concept map

A **token id** is an integer that names one entry in a tokenizer's vocabulary, and it is the only thing a language model actually receives. It relates to the **embedding table** by indexing it: the model looks up row number `id` to get the vector it computes with. It relates to **text** through the tokenizer, a fixed table learned once from training text that cuts a string into pieces and maps each piece to an id.

```text
text --tokenizer--> pieces --vocabulary--> ids --embedding table--> vectors --> model
```

Three choices of piece exist. Characters give a tiny table but long sequences. Whole words give short sequences but a table that never ends, because every typo, name and new coinage needs its own row. Subword tokens are the compromise, and byte-pair encoding (BPE) is the method this notebook builds. The key distinction from a dictionary: what counts as one token is a fact about the training text, not about language.

This matters to a practitioner because context limits, price and latency are all counted in tokens, not words. Check your provider's current pricing page for rates; none are stated here.

PART 1 - Attribute table

| Axis | Characters | Whole words | BPE tokens |
|---|---|---|---|
| Vocabulary size | small and fixed (an alphabet) | grows without bound as new words appear | chosen by the builder: base symbols plus one row per merge |
| Sequence length for the same text | longest | shortest | in between |
| Word never seen in training | always representable | needs an unknown marker, so information is lost | split into known pieces |
| Learned from data | no | yes (the word list) | yes (the ordered merge list) |
| Shares structure between related words | only at letter level | no: walk and walked are unrelated rows | yes: frequent chunks are reused |
| Unit that context limit, price and latency are counted in | no | no | yes |

PART 1 - Anchoring example

Neural machine translation hit this problem first. A fixed word-level vocabulary could not hold rare words, names and compounds, which left an open-vocabulary problem: what to do with words the system had never seen. Sennrich, Haddow and Birch (2016) proposed representing such words as sequences of subword units, so rare and unseen words become known pieces instead of unknowns. Byte-pair encoding, in several variants, is now a common way to build the tokenizers that sit in front of LLMs, which is why the token counts you see in LLM tools behave the way they do.

Source: [Sennrich, Haddow and Birch, Neural Machine Translation of Rare Words with Subword Units, ACL 2016](https://aclanthology.org/P16-1162/) - retrieved 2026-10-16.

PART 1 - What this teaches

- Recognize that the model's input is a sequence of integers, and that the embedding table is indexed by them
- Differentiate characters, whole words and subword tokens on the axes of vocabulary size and sequence length
- Identify which of the three has no upper bound on its vocabulary as new text keeps arriving

PART 1 - Try it

The next two cells cut one paragraph into characters and words, then give every distinct word an integer, which is what a word-level model would receive. Watch how a few misspellings of one name inflate that table. The third length, tokens, waits for Part 4 because it needs a tokenizer.

<strong style="color:red">KEY CONSIDERATION:</strong> context, price and latency are all counted in tokens, so whatever cuts text into more pieces costs more.

In [ ]:
# Same paragraph, cut up in different ways. PARAGRAPH is reused in Part 4 as the held-out text.
PARAGRAPH = (
    'Every prompt is cut into tokens before the model reads it. '
    'A rare name such as Quillfeather is split into more pieces than a common word such as water, '
    'and every piece counts against the context window and the bill.'
)


def normalize(text):
    # Lowercase letters only. This keeps the lesson about merging, not about punctuation rules.
    return re.findall(r'[a-z]+', text.lower())


words = normalize(PARAGRAPH)
print(f'characters : {len(PARAGRAPH)}')
print(f'words      : {len(words)}')
print('tokens     : measured in Part 4, once we have a tokenizer')


In [ ]:
# The model never sees strings: give every distinct word the next free integer.
word_to_id = {}
for w in words:
    word_to_id.setdefault(w, len(word_to_id))
ids = [word_to_id[w] for w in words]
print(ids)

size_before = len(word_to_id)
for typo in ['quilfeather', 'quillfether', 'quillfeathr']:
    word_to_id.setdefault(typo, len(word_to_id))  # each misspelling is a brand-new row
print(f'word vocabulary: {size_before} -> {len(word_to_id)} after three misspellings of one name')


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Scenario extension (Part 1):** A support team feeds a tokenizer a new batch of customer messages every day for a year. The messages are full of new product names, usernames and typos. The team keeps four running tables: the distinct characters seen, the distinct whole words seen, a BPE vocabulary trained once with a fixed number of merges, and the distinct byte values seen.
>
> **As the batches keep arriving, which table has no upper limit?**
>
> (a) the distinct characters seen
> (b) the distinct whole words seen
> (c) the BPE vocabulary trained with a fixed number of merges
> (d) the distinct byte values seen

<br>

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (b) - (a) loses because characters come from a finite writing system, so the table saturates. (c) loses because its size is fixed by construction at base symbols plus N merges, however much text arrives. (d) loses because a byte takes only 256 values. (b) wins because every new name, username and typo is a word never seen before, so the word table gains a row for each and never stops. -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## Counting **pairs** and merging

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/01_02_diagram.svg" align="center" width="50%" padding="10"><br>
    <br>
</div>

PART 2 - Concept map

**Byte-pair encoding (BPE)** is a procedure that builds a tokenizer's vocabulary by repeatedly merging the most frequent adjacent pair of symbols in the training text into one new symbol. It relates to **data compression** by sharing its core move: Gage (1994) published the compression algorithm the method is named after, and BPE reuses the idea of replacing a frequent pair with a new symbol to decide which strings earn a vocabulary entry. It relates to the **vocabulary** by growing it one entry per merge, starting from single characters.

The procedure, in four steps:

1. **Start.** Split every word into characters and add an end-of-word marker `</w>`, so that `er` inside a word can be told apart from `er` at the end of one.
2. **Count.** For every adjacent pair of symbols, add up how often it occurs across the corpus, weighting each word by how many times it appears.
3. **Merge.** Take the pair with the highest count (ties go to the alphabetically first pair, so every run gives the same table), fuse it into a new symbol everywhere, and record the merge.
4. **Repeat** steps 2 and 3 for N merges. The ordered list of merges is the tokenizer.

The key distinction from a word list: nobody chose which strings get a row. Frequency in the training text decided it.

Gage's original is a compression step: find the most frequent pair of adjacent bytes, replace every instance with a byte that does not occur in the data, repeat until no frequent pair or no unused byte is left, and write the substitution table ahead of the packed data so the expander can undo it. Sennrich, Haddow and Birch (2016) adapted it to words: start from characters plus an end-of-word symbol, and merge the most frequent pair into a new symbol instead of a spare byte. That adaptation is the procedure above.

PART 2 - Attribute table

| Axis | Start (0 merges) | After N merges |
|---|---|---|
| Symbols per word | one per letter, plus the end marker | fewer and longer |
| Vocabulary size | distinct characters plus the end marker | that, plus exactly N merged symbols |
| Most frequent word | many symbols | eventually a single symbol |
| What dominates the pair counts | common letter pairs | pairs that include already-merged symbols |
| What the tokenizer knows | nothing about words | which chunks were frequent in the training text |
| Table rows added per step | none | exactly one per merge |

PART 2 - Anchoring example

The Hugging Face LLM course works a corpus of five words: hug (10 times), pug (5), pun (12), bun (4) and hugs (5). Starting from characters, the pair (u, g) is the most frequent, with 20 occurrences across hug, pug and hugs, so it merges into the new symbol ug. The next merges the course lists are (u, n) into un, and then (h, ug) into hug. Notice that the third merge builds on the first: merged symbols become parts of later pairs, which is how short pieces grow into long ones. The course does not use an end-of-word marker; this notebook adds one.

Source: [Hugging Face LLM course, chapter 6, byte-pair encoding tokenization](https://huggingface.co/learn/llm-course/chapter6/5) - retrieved 2026-10-16.

PART 2 - What this teaches

- Recognize that each BPE merge fuses the single most frequent adjacent pair and adds exactly one entry to the vocabulary
- Identify the four steps (start, count, merge, repeat) and which of them uses word frequency
- Differentiate a merge that builds on an earlier merge from one that fuses two plain characters

PART 2 - Try it

The helpers in the first cell are reused in Parts 3 and 4, so they live here. Read `pair_counts` and `merge_symbols` first; everything else is a loop around them. The second cell builds a 30-word toy corpus and prints its starting pair counts. The third trains up to 40 merges and prints the merge table in three slices (merges 1-10, 11-20, 21-40).

If fewer than 40 merges appear, the corpus ran out of pairs: every word has become a single symbol. A 30-word corpus has only a handful of distinct words, so the tokenizer ends up memorizing them. That is a toy-corpus effect, and Part 4 uses a longer text to avoid it.

In [ ]:
END = '</w>'  # end-of-word marker, one extra symbol per word


def to_symbols(word):
    return tuple(word) + (END,)


def build_corpus(text):
    # {symbol tuple: number of times the word occurs}. A dict, so each distinct word is merged once.
    return {to_symbols(w): n for w, n in Counter(normalize(text)).items()}


def pair_counts(corpus):
    counts = Counter()
    for symbols, freq in corpus.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += freq  # weight by word frequency
    return counts


def best_pair(counts):
    # Highest count wins; ties go to the alphabetically first pair so every run gives the same table.
    return min(counts, key=lambda pair: (-counts[pair], pair))


def merge_symbols(symbols, pair):
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2  # skip both halves: one symbol cannot take part in two merges at once
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)


def merge(corpus, pair):
    merged = {}
    for symbols, freq in corpus.items():
        new = merge_symbols(symbols, pair)
        merged[new] = merged.get(new, 0) + freq
    return merged


def train_bpe(text, num_merges):
    corpus = build_corpus(text)
    merges = []  # (pair, count) in the order learned. This ordered list IS the tokenizer.
    for _ in range(num_merges):
        counts = pair_counts(corpus)
        if not counts:
            break  # every word is already a single symbol, nothing is left to merge
        pair = best_pair(counts)
        merges.append((pair, counts[pair]))
        corpus = merge(corpus, pair)
    return merges, corpus


In [ ]:
TOY_TEXT = (
    'low low low low low lower lower newest newest newest newest newest newest '
    'widest widest widest wide wide new new new lowest lowest low newer newer '
    'wider wider newest new'
)
assert len(normalize(TOY_TEXT)) == 30, 'the toy corpus should have exactly 30 words'

corpus = build_corpus(TOY_TEXT)
for symbols, freq in corpus.items():
    spaced = ' '.join(symbols)
    print(f'{freq:>3} x {spaced}')

print()
print('top 5 pairs before any merge:')
for pair, n in sorted(pair_counts(corpus).items(), key=lambda kv: (-kv[1], kv[0]))[:5]:
    print(f'  {pair} -> {n}')


In [ ]:
merges, final_corpus = train_bpe(TOY_TEXT, 40)


def show_merges(merges, start, stop):
    for i, (pair, count) in enumerate(merges[start:stop], start=start + 1):
        print(f'{i:>3}. {pair[0]!r} + {pair[1]!r} -> {(pair[0] + pair[1])!r}  (count {count})')


previous = 0
for checkpoint in (10, 20, 40):
    learned = min(checkpoint, len(merges))
    print(f'--- merge table at {checkpoint} requested ({learned} learned so far) ---')
    show_merges(merges, previous, learned)
    previous = learned

print()
print('final corpus, each word as the symbols left after all merges:')
for symbols, freq in final_corpus.items():
    spaced = ' '.join(symbols)
    print(f'{freq:>3} x {spaced}')

total_before = sum(len(s) * n for s, n in corpus.items())
total_after = sum(len(s) * n for s, n in final_corpus.items())
print(f'symbols in the corpus: {total_before} -> {total_after}')


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Scenario extension (Part 2):** A teammate trains BPE on eight words: cat four times, cap once and mat three times, each with the end marker, and ties go to the alphabetically first pair (the rule in `best_pair`). They report that the first merge is (c, a) because cat is the most frequent word.
>
> **Which pair actually merges first?**
>
> (a) (c, a)
> (b) (a, t)
> (c) (t, </w>)
> (d) (m, a)

<br>

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (b) - Pair counts add up every word that contains the pair: (c, a) = 4 + 1 = 5, (a, t) = 4 + 3 = 7, (t, </w>) = 4 + 3 = 7, (m, a) = 3. (a) loses because the most frequent word does not decide the merge; pair totals across all words do. (d) loses with only 3. (c) ties with (a, t) at 7, but the tie goes to the alphabetically first pair and a comes before t, so (a, t) wins. -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **Encoding** and decoding

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/01_03_diagram.svg" align="center" width="50%" padding="10"><br>
    <br>
</div>

PART 3 - Concept map

**Encoding** applies the learned merges, in the order they were learned, to new text: split each word into characters plus the end marker, run merge 1, then merge 2, and so on, fusing any adjacent pair that matches. **Decoding** goes the other way and is only a lookup: each id names one piece, and joining the pieces (turning the end marker back into a space) rebuilds the text. Encoding relates to training by replaying it on text the tokenizer has not seen; the fixed merge order is what makes the result deterministic. The key distinction from a word lookup: an unseen word is not an error, it simply ends up as smaller pieces that were seen.

Decoding round-trips exactly as long as every piece is in the vocabulary. The exception is a character the tokenizer never saw: it maps to an unknown id, and that id cannot be turned back into the original character.

PART 3 - Attribute table

| Axis | Word seen in training | Unseen word, known characters | Word with characters never seen |
|---|---|---|---|
| Pieces produced | often one large piece | known sub-pieces, down to single characters if nothing else matches | known pieces, plus an unknown marker per missing character |
| Tokens used | fewest | more | more still |
| Round trip through decode | exact | exact | lossy: unknown ids cannot be restored |
| Why it happens | its merges were all learned | the merges that fire are the ones its parts share with the corpus | no merge can touch a character that is not in the base set |
| Remedy | none needed | none needed | start from bytes instead of characters, as byte-level BPE does |

PART 3 - Anchoring example

The Hugging Face course tokenizes new words with the five-word vocabulary from Part 2. Merges are applied in learned order, so bug becomes b and ug. The word mug becomes an unknown marker followed by ug, because the letter m never appeared in the training corpus and so is not in the base vocabulary. The word thug becomes an unknown marker followed by hug. The course also notes that GPT-2 and RoBERTa avoid unknowns by using byte-level BPE, which starts from a base vocabulary of 256 byte values.

Source: [Hugging Face LLM course, chapter 6, byte-pair encoding tokenization](https://huggingface.co/learn/llm-course/chapter6/5) - retrieved 2026-10-16.

PART 3 - What this teaches

- Recognize that encoding replays the learned merges in their original order
- Identify the boundaries where an unseen word splits into known sub-pieces
- Differentiate a lossless round trip (every character known) from a lossy one (unknown ids present)

PART 3 - Try it

The first cell defines `encode` (text to pieces), `to_ids` (pieces to integers) and `decode` (integers back to text), and builds the vocabulary from the Part 2 merges. The second cell encodes a sentence with one common word and one word the corpus never contained, prints the pieces and then the ids, and asserts that decoding restores the input. The third cell compares three words, so you can read where each one splits.

One simplification to keep in mind: `normalize` keeps lowercase letters only, so punctuation and capitalization are dropped before encoding.

In [ ]:
UNK = '<unk>'  # id used for any piece that is not in the vocabulary


def encode(text, merges):
    # Returns the list of pieces. Merges run in learned order: later merges build on earlier ones.
    pair_list = [pair for pair, _ in merges]
    pieces = []
    for word in normalize(text):
        symbols = to_symbols(word)
        for pair in pair_list:
            symbols = merge_symbols(symbols, pair)
        pieces.extend(symbols)
    return pieces


def build_vocab(text, merges):
    base = sorted(set(''.join(normalize(text))) | {END})
    symbols = [UNK] + base + [pair[0] + pair[1] for pair, _ in merges]
    vocab = {}
    for sym in symbols:
        vocab.setdefault(sym, len(vocab))  # keeps ids dense if two merges ever build the same string
    return vocab


def to_ids(pieces, vocab):
    return [vocab.get(p, vocab[UNK]) for p in pieces]


def decode(ids, vocab):
    id_to_piece = {i: p for p, i in vocab.items()}
    return ''.join(id_to_piece[i] for i in ids).replace(END, ' ').strip()


In [ ]:
vocab = build_vocab(TOY_TEXT, merges)
print(f'vocabulary size: {len(vocab)}')

sentence = 'newest slower'  # one common word, one word the corpus never contained
pieces = encode(sentence, merges)
ids = to_ids(pieces, vocab)
print('pieces :', pieces)
print('ids    :', ids)

restored = decode(ids, vocab)
print('decoded:', restored)
assert restored == ' '.join(normalize(sentence)), 'round trip failed'


In [ ]:
for word in ['newest', 'slower', 'lowered']:
    word_pieces = encode(word, merges)
    print(f'{word:<8} -> {word_pieces}   ids {to_ids(word_pieces, vocab)}')
print(f'the unknown id is {vocab[UNK]}')


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Scenario extension (Part 3):** The tokenizer from this Part is deployed unchanged, with the vocabulary built by `build_vocab`. A user types the word kayak. None of the letters k, a or y appear in the 30-word training corpus. The pipeline log shows the model received six ids for that single word.
>
> **What are those six ids?**
>
> (a) one new id for the whole word kayak, added to the vocabulary when it is first seen
> (b) the unknown id five times, followed by the id of the end-of-word marker
> (c) the unknown id six times, one for every piece including the end marker
> (d) six single-character ids, because single characters are always in the vocabulary

<br>

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (b) - The word splits into k, a, y, a, k and the end marker, six pieces. (a) loses because the vocabulary is fixed after training and encoding never adds to it. (d) loses because the base set holds only characters seen in the training text, and k, a and y were not. (c) loses because build_vocab adds the end marker to the base set explicitly, so it is known even when every letter is not. (b) wins: five letters fall back to the unknown id and the marker keeps its own id. -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **Vocabulary size** against sequence length

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/01_04_diagram.svg" align="center" width="50%" padding="10"><br>
    <br>
</div>

PART 4 - Concept map

**Vocabulary size** is the number of distinct pieces a tokenizer can emit: its base symbols plus one entry per merge. It relates to **sequence length** by a trade: each extra merge can only fuse adjacent symbols, so the same text never gets longer and usually gets shorter. The gains shrink because merges are learned most-frequent first. Early merges cover pairs that occur everywhere; later merges cover rarer and rarer pairs, so each new row saves fewer positions than the one before. The key distinction from a pure compression view: a bigger vocabulary is not free, because every row is a vector the model must store and learn, and a row that is rarely seen is learned poorly.

Shorter sequences are the payoff, since context, price and latency are all counted in tokens. The sweep below measures the payoff directly.

PART 4 - Attribute table

| Axis | Small vocabulary | Large vocabulary |
|---|---|---|
| Sequence length for the same text | longer | shorter |
| Rows in the embedding table | few | many |
| Training signal per row | each row is seen often | many rows are seen rarely |
| Cost per prompt (counted in tokens) | more tokens | fewer tokens |
| Saving from the next merge | large at first | small after many merges |
| Limiting case | characters only: longest sequences | whole words: shortest sequences, but the table cannot cover new words |

PART 4 - Anchoring example

In the Hugging Face walkthrough, the starting vocabulary is the set of symbols that appear in the corpus (seven letters in the toy example), and every merge adds exactly one new token to it. For GPT-2 and RoBERTa, the course says the base vocabulary is instead the 256 byte values. So a BPE vocabulary has size base plus merges, and the number of merges is the knob a trainer turns. The cells below turn that knob from 0 to 200 and watch the length of one fixed paragraph.

Source: [Hugging Face LLM course, chapter 6, byte-pair encoding tokenization](https://huggingface.co/learn/llm-course/chapter6/5) - retrieved 2026-10-16.

Measured in `02_three_real_tokenizers.ipynb`: GPT-2's vocabulary has 50,257 entries and `cl100k_base` has 100,277, so production vocabularies sit in the tens of thousands to about a hundred thousand, two to three orders of magnitude above the 228 learned here.

PART 4 - What this teaches

- Recognize that more merges mean a larger table and shorter sequences, with diminishing savings per merge
- Identify the vocabulary size at each point on the curve as base symbols plus the number of merges
- Differentiate the benefit of a bigger vocabulary (fewer tokens per prompt) from its cost (more rows to store and learn)

PART 4 - Try it

The first cell trains up to 200 merges on a longer training paragraph, then encodes the held-out paragraph from Part 1 with the first 0, 5, 10, ... merges, recording vocabulary size and length each time. If the training text cannot support 200 merges, the loop uses as many as exist. The second cell plots the curve to `images/01_vocab_vs_length.png` and closes the loop on Part 1 with the three lengths side by side.

Two caveats on the numbers. Token counts here include one end-of-word marker per word, which production tokenizers handle differently. And the comparison counts letters, not the punctuation and spaces in the raw paragraph, so treat it as a ballpark.

In [ ]:
TRAIN_TEXT = (
    'a tokenizer reads text and cuts it into pieces before a language model sees anything. '
    'each piece is mapped to a number and the model works only with those numbers. '
    'common words such as the and water are usually kept whole because they appear so often. '
    'rare words are broken into smaller parts that are shared with other words. '
    'this keeps the table of pieces small and still lets the model read new names and misspellings. '
    'the table is built once from a large collection of text and then it never changes. '
    'when a team writes a long prompt the tokenizer decides how many pieces it becomes and the provider counts those pieces. '
    'a longer prompt means more pieces and more pieces mean more time and more money. '
    'the same sentence can cost a different amount in another language because the table was built from different text. '
    'a good tokenizer finds a balance between a table that is too small and sequences that are too long '
    'and a table that is too big and rarely used. '
    'numbers and code and unusual spellings often need many small pieces. '
    'quick experiments with jazz lyrics and zebra names and extra vowels show the effect. '
    'engineers who understand this can shorten prompts and predict costs before sending a request.'
)

merges_big, _ = train_bpe(TRAIN_TEXT, 200)
print(f'merges learned: {len(merges_big)}')

# Merge counts to test: every 5th merge, plus the final one if it falls between steps.
steps = list(range(0, len(merges_big) + 1, 5))
if steps[-1] != len(merges_big):
    steps.append(len(merges_big))

vocab_sizes = [len(build_vocab(TRAIN_TEXT, merges_big[:n])) for n in steps]
lengths = [len(encode(PARAGRAPH, merges_big[:n])) for n in steps]

# More merges can only fuse symbols, never split them, so length must never rise.
assert all(a >= b for a, b in zip(lengths, lengths[1:])), 'length rose with more merges'

missing = sorted(set(''.join(normalize(PARAGRAPH))) - set(''.join(normalize(TRAIN_TEXT))))
print(f'letters in the held-out paragraph never seen in training: {missing}')

print('merges   vocab   length')
for n, v, length in list(zip(steps, vocab_sizes, lengths))[::4]:
    print(f'{n:>6} {v:>7} {length:>8}')

gains = [lengths[i] - lengths[i + 1] for i in range(len(lengths) - 1)]
print(f'tokens saved by merges {steps[0]}-{steps[1]}: {gains[0]}')
print(f'tokens saved by merges {steps[-2]}-{steps[-1]}: {gains[-1]}')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(vocab_sizes, lengths, marker='o', linewidth=2, color='#003262')
ax.set_xlabel('vocabulary size (distinct symbols)')
ax.set_ylabel('tokens for the held-out paragraph')
ax.set_title('More merges, shorter sequences, flattening returns')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('images/01_vocab_vs_length.png', dpi=150)
plt.show()
assert os.path.exists('images/01_vocab_vs_length.png'), 'plot was not saved'

n_letters = sum(len(w) for w in normalize(PARAGRAPH))
n_words = len(normalize(PARAGRAPH))
print(f'letters {n_letters} | words {n_words} | tokens after {steps[-1]} merges {lengths[-1]} | tokens at 0 merges {lengths[0]}')


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Scenario extension (Part 4):** A team proposes growing the vocabulary until almost every common word and name is a single token, and asks for a table with millions of entries. Their own sweep looks like the plot above, with sequence length still creeping down at the right edge. Their trial model now has a huge embedding table, and most rows are seen only a handful of times in training.
>
> **What is the strongest reason production vocabularies stop at a large but bounded size instead of growing to millions?**
>
> (a) Past a point each extra entry saves almost no tokens, yet still costs a stored vector and training signal, and rarely seen entries are learned poorly
> (b) BPE cannot learn more merges than there are characters in the alphabet
> (c) A larger vocabulary makes encoded sequences longer, which raises the price per prompt
> (d) The merge loop becomes too slow to run even once, so larger tables cannot be trained

<br>

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (a) - (b) loses because the number of merges is bounded by the training text, not by the alphabet; the sweep above learned 200 merges on a text that starts from 28 symbols, so most of the vocabulary is merged pieces. (c) loses because a merge can only fuse symbols, so more merges never lengthen a sequence. (d) loses because training runs once, offline, so its speed is a one-time cost and not the reason for a ceiling. (a) wins because the savings per row fall toward zero while the cost per row (a stored vector that needs enough examples to be learned) does not. -->

<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

## Sources Consulted

- [https://aclanthology.org/P16-1162/](https://aclanthology.org/P16-1162/) - retrieved 2026-10-16. Section 3.2 read in full: the character-level adaptation with the end-of-word symbol and the iterative merge of the most frequent pair.
- [https://www.derczynski.com/papers/archive/BPE_Gage.pdf](https://www.derczynski.com/papers/archive/BPE_Gage.pdf) - retrieved 2026-10-16. Section 1, Theory, read in full: replace the most frequent adjacent byte pair with an unused byte, repeat, write the substitution table before the data.
- [https://huggingface.co/learn/llm-course/chapter6/5](https://huggingface.co/learn/llm-course/chapter6/5) - retrieved 2026-10-16

<hr style="border: 6px solid#003262;" />